[Guide Index](README.md) | [Appendix A](appendix-a.ipynb)


# Appendix A3: A Complex DL Reasoning Example - Constraint Satisfaction at Scale

The [OWL 2 DL Reasoning guide](02c-owl-dl-reasoning.ipynb) keeps its own examples small and focused on the core ideas - disjunctive class reasoning, class unsatisfiability, consistency checking. This appendix is the one example from that guide big enough to need its own space: encoding a full `N=20` employee/job assignment problem as an OWL 2 DL ontology and solving it via consistency-checking alone - forced pairings found with zero search, the rest resolved by a hand-rolled guess/check/backtrack loop, an unsolvable variant, random-instance generation, and a look at why this approach doesn't scale past a few dozen individuals.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later cells reuse variables from earlier ones.

`owlready2` additionally requires a Java runtime on `PATH` — HermiT is a Java program. If Java isn't found or isn't configured correctly, an error message is raised.

In [1]:
from starlayer import StarLayerGraph
from starlayer.graph import ENTAILMENT, Namespace, RDF

EX = Namespace("http://example.org/")

## 1. Solving a constraint-satisfaction problem: employee/job assignment

`N=20` employees, `N=20` jobs, every employee restricted to a subset of at least two jobs they `canDo`. Every employee must end up assigned to exactly one job they can do, and no two employees share a job. This is the same "inject `N` distinct individuals into an `N`-member `owl:oneOf` set" bijection trick as [section 1.2 in the OWL 2 DL Reasoning guide](02c-owl-dl-reasoning.ipynb#1.2-A-small-logic-puzzle) - `assignedTo` declared both `owl:FunctionalProperty` and `owl:InverseFunctionalProperty`, with every employee and every job pairwise `owl:differentFrom` - generalized one step further: instead of every individual sharing *one* common candidate set, **each employee gets its own `someValuesFrom` restriction naming only its own `canDo` jobs**. `assignedTo rdfs:subPropertyOf canDo` means any assignment HermiT commits to is automatically entailed as a `canDo` fact too, for free.

The employee/job names and all the `owl:differentFrom`/`someValuesFrom`/`oneOf` boilerplate are generated from `N` and a `canDo` **JSON object** rather than hand-typed (1.1). The `canDo` object used for this walkthrough is itself a **pinned random draw** - one specific output of the random generator introduced in 1.5, captured once (`random.Random(42)`, seed fixed for reproducibility) and hard-coded here so the notebook always demonstrates the exact same forced pairings and solution rather than a fresh shuffle every run. Nothing about which pairings end up forced, or which employees end up causing the unsolvable case in 1.4, was hand-picked - that structure is wherever the random construction happened to put it, not authored for effect.

**A note on scale and timing:** `N=20` was chosen deliberately, not arbitrarily - reasoning time for this encoding grows roughly *exponentially* with `N`, not smoothly. A quick empirical check before settling on 20: `N=10` reasoned in well under a second, `N=20` takes a few seconds per check - already close to an order-of-magnitude jump for doubling `N`. This isn't a defect in the encoding so much as an inherent property of the OWL constructs the problem needs (`owl:oneOf` nominals plus pairwise `owl:differentFrom` plus `Functional`/`InverseFunctional` together are a textbook expensive combination for tableau reasoners) - see 1.6 for the actual numbers and what they mean. Each `infer()` call below prints its own elapsed time, so the notebook's own output is the record of "how long this actually took," not a claim in this text.

### 1.1 Generate the ontology from `N` and a `canDo` JSON object

`make_entities(n, prefix)` produces the `N` employee/job names; `build_assignment_ontology(can_do)` derives everyone's `owl:differentFrom` pairs, `canDo` facts, and per-employee `someValuesFrom`/`oneOf` restriction straight from the `canDo` dict's own keys and values - unchanged from before, just fed a larger `N` this time.

In [2]:
import json

def make_entities(n, prefix):
    """N entity names like EMP01..EMP05 or Job01..Job05."""
    width = max(2, len(str(n)))
    return [f"{prefix}{i:0{width}d}" for i in range(1, n + 1)]


def build_assignment_ontology(can_do):
    """Turtle text for an assignment-problem ontology, generated from a
    {employee: [job, ...]} JSON object - canDo facts, plus the OWL 2
    machinery (assignedTo Functional+InverseFunctional+subPropertyOf
    canDo, one someValuesFrom/oneOf restriction per employee, everyone
    pairwise differentFrom) that turns it into a bijection problem."""
    employees = list(can_do)
    jobs = sorted({j for opts in can_do.values() for j in opts})

    def pairwise_different(names):
        lines = [
            f"ex:{a} owl:differentFrom {', '.join('ex:' + b for b in names[i + 1:])} ."
            for i, a in enumerate(names) if names[i + 1:]
        ]
        return "\n".join(lines)

    employee_types = "\n".join(f"ex:{e} a ex:Employee ." for e in employees)
    can_do_facts = "\n".join(
        f"ex:{e} ex:canDo {', '.join('ex:' + j for j in opts)} ."
        for e, opts in can_do.items()
    )
    restrictions = "\n".join(
        f"ex:{e} a [ a owl:Restriction ; owl:onProperty ex:assignedTo ;\n"
        f"             owl:someValuesFrom [ owl:oneOf ( {' '.join('ex:' + j for j in opts)} ) ] ] ."
        for e, opts in can_do.items()
    )

    return f"""
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:canDo a owl:ObjectProperty .
ex:assignedTo a owl:ObjectProperty, owl:FunctionalProperty, owl:InverseFunctionalProperty ;
    rdfs:subPropertyOf ex:canDo .

{employee_types}
{pairwise_different(employees)}

{pairwise_different(jobs)}

{can_do_facts}

{restrictions}
"""

In [3]:
from starlayer.graph.graph.owl_dl import InconsistentOntologyError

EMPLOYEES = make_entities(20, "EMP")
JOBS = make_entities(20, "Job")

# A pinned random draw from random_solvable_can_do(20, random.Random(42)) -
# see 1.5. Hard-coded rather than generated live so this walkthrough is
# always the same instance, but the *pattern* (who's forced, in what order,
# how much search is needed) was never hand-picked - it's whatever that
# draw happened to produce.
CAN_DO = {
    "EMP01": ["Job14", "Job18", "Job20"],
    "EMP02": ["Job06", "Job10", "Job16"],
    "EMP03": ["Job06", "Job15"],
    "EMP04": ["Job05", "Job06", "Job08", "Job10", "Job12"],
    "EMP05": ["Job03", "Job04", "Job10", "Job14"],
    "EMP06": ["Job12", "Job14"],
    "EMP07": ["Job02", "Job09", "Job15", "Job16"],
    "EMP08": ["Job13", "Job19"],
    "EMP09": ["Job07", "Job19"],
    "EMP10": ["Job03", "Job07", "Job12", "Job13"],
    "EMP11": ["Job08", "Job18"],
    "EMP12": ["Job03", "Job04", "Job08", "Job11"],
    "EMP13": ["Job02", "Job07", "Job10", "Job13", "Job16"],
    "EMP14": ["Job07", "Job09", "Job12", "Job13"],
    "EMP15": ["Job03", "Job07"],
    "EMP16": ["Job06", "Job15", "Job17"],
    "EMP17": ["Job08", "Job09", "Job10", "Job12", "Job19"],
    "EMP18": ["Job08", "Job09"],
    "EMP19": ["Job01", "Job12"],
    "EMP20": ["Job03", "Job04", "Job08", "Job10", "Job12"]
}

assert set(CAN_DO) == set(EMPLOYEES)
assert set().union(*CAN_DO.values()) <= set(JOBS)

BASE_TTL = build_assignment_ontology(CAN_DO)
print(f"{len(EMPLOYEES)} employees, {len(JOBS)} jobs, ontology text: {len(BASE_TTL):,} characters")

20 employees, 20 jobs, ontology text: 9,643 characters


### 1.2 Reduce: what's forced with zero assumptions

At `N=20` with a real random `canDo` spread, it's rarely just one employee that's trivially pinned down - several jobs typically turn out to be reachable by only one employee each, so several assignments come back forced simultaneously, each for the same reason as before: not a giveaway restriction on that one employee, but a consequence of the shape of the whole bipartite graph. HermiT finds all of them in a single `infer()` call, with no search.

In [4]:
import time

def local(uri):
    return str(uri).rsplit("/", 1)[1]

g0 = StarLayerGraph()
g0.bind("ex", EX)
g0.parse(data=BASE_TTL, format="turtle12")

t0 = time.time()
closed0 = g0.infer(entailment=ENTAILMENT['OWL-Direct'])
t_forced = time.time() - t0

committed = {local(s): local(o) for s, p, o in closed0.triples((None, EX.assignedTo, None))}
print(f"Forced by inference alone ({len(committed)} pairings, {t_forced:.2f}s): {committed}")

Forced by inference alone (6 pairings, 1.23s): {'EMP16': 'Job17', 'EMP19': 'Job01', 'EMP01': 'Job20', 'EMP12': 'Job11', 'EMP04': 'Job05', 'EMP11': 'Job18'}


### 1.3 Search the rest: guess, check, backtrack

For each remaining employee, try each of its candidate jobs in turn (skipping any job already taken - `assignedTo` being `InverseFunctionalProperty` means committing two employees to the same job is always inconsistent, so there's no point spending a reasoner call to discover that). Build a fresh graph with every *committed* `assignedTo` fact plus the tentative new one, and ask HermiT whether that's still consistent. Consistent → keep it and recurse on the rest; inconsistent → undo and try the next candidate. Each candidate check is its own `infer()` call, so this is where most of the wall-clock time in this section goes - the cell prints how many calls it took and how long.

In [5]:
def is_consistent(committed_assignments):
    facts = "\n".join(f"ex:{e} ex:assignedTo ex:{j} ." for e, j in committed_assignments.items())
    g = StarLayerGraph()
    g.parse(data=BASE_TTL + "\n" + facts, format="turtle12")
    try:
        g.infer(entailment=ENTAILMENT['OWL-Direct'])
        return True
    except InconsistentOntologyError:
        return False


call_count = 0

def backtrack(remaining):
    global call_count
    if not remaining:
        return True
    emp = remaining[0]
    for job in CAN_DO[emp]:
        if job in committed.values():
            continue
        committed[emp] = job
        call_count += 1
        if is_consistent(committed):
            if backtrack(remaining[1:]):
                return True
        else:
            print(f"  {emp} -> {job}: inconsistent, backtracking")
        del committed[emp]
    return False


remaining = [e for e in CAN_DO if e not in committed]
t0 = time.time()
solved = backtrack(remaining)
t_search = time.time() - t0

print(f"\nSolved: {solved} ({call_count} infer() calls, {t_search:.2f}s)")
print("Full assignment:", committed)
assert solved and len(committed) == 20 and len(set(committed.values())) == 20

  EMP06 -> Job12: inconsistent, backtracking


  EMP09 -> Job07: inconsistent, backtracking


  EMP10 -> Job07: inconsistent, backtracking


  EMP13 -> Job07: inconsistent, backtracking


  EMP13 -> Job10: inconsistent, backtracking


  EMP14 -> Job07: inconsistent, backtracking


  EMP17 -> Job08: inconsistent, backtracking



Solved: True (21 infer() calls, 10.05s)
Full assignment: {'EMP16': 'Job17', 'EMP19': 'Job01', 'EMP01': 'Job20', 'EMP12': 'Job11', 'EMP04': 'Job05', 'EMP11': 'Job18', 'EMP02': 'Job06', 'EMP03': 'Job15', 'EMP05': 'Job03', 'EMP06': 'Job14', 'EMP07': 'Job02', 'EMP08': 'Job13', 'EMP09': 'Job19', 'EMP10': 'Job12', 'EMP13': 'Job16', 'EMP14': 'Job09', 'EMP15': 'Job07', 'EMP17': 'Job10', 'EMP18': 'Job08', 'EMP20': 'Job04'}


### 1.4 When there's no solution at all

Same `build_assignment_ontology()`, same `N=20` - only the `canDo` JSON object differs, and it's a second pinned random draw (`random_unsolvable_can_do(20, random.Random(42))`, 1.5), not a hand-picked scenario. Whatever subset and jobs the random construction happened to violate Hall's theorem with is what's below - in this draw, 8 employees all restricted to the exact same 6 jobs, which 8 and which 6 isn't something worth memorizing, just something the construction guarantees is impossible to staff. `InconsistentOntologyError` fires on the *whole* problem - HermiT doesn't need a search to find this, it falls straight out of the base constraints, the same way several pairings fell out in 1.2 without one.

In [6]:
CAN_DO_UNSOLVABLE = {
    "EMP01": ["Job05", "Job16", "Job17"],
    "EMP02": ["Job02", "Job04"],
    "EMP03": ["Job06", "Job09", "Job13", "Job14", "Job18", "Job20"],
    "EMP04": ["Job06", "Job09", "Job13", "Job14", "Job18", "Job20"],
    "EMP05": ["Job06", "Job09", "Job13", "Job14", "Job18", "Job20"],
    "EMP06": ["Job03", "Job06", "Job14"],
    "EMP07": ["Job01", "Job09", "Job13", "Job15", "Job17"],
    "EMP08": ["Job09", "Job18"],
    "EMP09": ["Job06", "Job09", "Job13", "Job14", "Job18", "Job20"],
    "EMP10": ["Job04", "Job06", "Job10", "Job14"],
    "EMP11": ["Job01", "Job04", "Job06", "Job09", "Job17"],
    "EMP12": ["Job06", "Job09", "Job13", "Job14", "Job18", "Job20"],
    "EMP13": ["Job06", "Job09", "Job13", "Job14", "Job18", "Job20"],
    "EMP14": ["Job05", "Job07", "Job12", "Job17"],
    "EMP15": ["Job06", "Job09", "Job13", "Job14", "Job18", "Job20"],
    "EMP16": ["Job06", "Job09", "Job13", "Job14", "Job18", "Job20"],
    "EMP17": ["Job01", "Job17", "Job18"],
    "EMP18": ["Job01", "Job04", "Job11", "Job12", "Job15", "Job16"],
    "EMP19": ["Job02", "Job03", "Job08", "Job20"],
    "EMP20": ["Job03", "Job16"]
}

g_stuck = StarLayerGraph()
g_stuck.parse(data=build_assignment_ontology(CAN_DO_UNSOLVABLE), format="turtle12")

t0 = time.time()
try:
    g_stuck.infer(entailment=ENTAILMENT['OWL-Direct'])
    status = "consistent (unexpected)"
except InconsistentOntologyError as e:
    status = f"InconsistentOntologyError (expected): {e}"
t_unsolvable = time.time() - t0

print(f"{status}\n({t_unsolvable:.2f}s)")

InconsistentOntologyError (expected): infer(entailment=ENTAILMENT['OWL-Direct']) found self's data logically inconsistent under OWL 2 DL semantics.
(0.59s)


### 1.5 Generating random solvable and unsolvable instances

The `canDo` objects used in 1.1 and 1.4 weren't hand-picked - they're the literal, unedited output of the two functions below, seeded with `random.Random(42)`. Regenerating them here with the same seed reproduces both exactly, which is checked below rather than just claimed.

- **Solvable:** plant a random bijection first (`random.sample` a full permutation of jobs onto employees) as a hidden ground-truth solution, then give each employee a few random extra options on top of their guaranteed job (capped, so `canDo` lists stay realistically sparse rather than "everyone can do almost everything" - a denser graph is not obviously easier for HermiT and was, in early testing, sometimes considerably worse). A valid assignment exists no matter what the extras are - the planted permutation itself is always one.
- **Unsolvable:** trigger a genuine [Hall's marriage theorem](https://en.wikipedia.org/wiki/Hall%27s_marriage_theorem) violation - pick a random subset of `k` employees and a random subset of `t` jobs with `t < k`, and restrict every employee in that subset to draw only from those `t` jobs. That subset alone can never get distinct jobs, so the whole problem is unsolvable regardless of what any other employee can do - the same construction 1.4 used, just with `k`/`t`/the subsets themselves chosen randomly instead of picked to make a clean example.

The batch loop below draws a handful of *additional* random instances of each kind (continuing the same `rng`, so distinct from the two already pinned above) through `infer()` and confirms HermiT's verdict matches what each construction guarantees, every time - a lightweight self-check that the whole encoding is sound, not just the one worked example. Kept deliberately small (5 of each, not 20) given each check now costs a few seconds at `N=20` rather than milliseconds at `N=5`.

In [7]:
import random


def random_solvable_can_do(n, rng, min_options=2, max_extra=4):
    """Plant a random bijection as a hidden solution, then add a capped
    number of random extra options per employee - guarantees at least one
    valid assignment exists no matter what those extras turn out to be."""
    employees = make_entities(n, "EMP")
    jobs = make_entities(n, "Job")
    permuted_jobs = jobs[:]
    rng.shuffle(permuted_jobs)

    can_do = {}
    for emp, guaranteed in zip(employees, permuted_jobs):
        pool = [j for j in jobs if j != guaranteed]
        extra_count = rng.randint(max(0, min_options - 1), min(max_extra, len(pool)))
        extras = rng.sample(pool, extra_count)
        can_do[emp] = sorted([guaranteed] + extras)
    return can_do


def random_unsolvable_can_do(n, rng, min_options=2, max_extra=4):
    """Pick a random employee subset of size k and a random job subset
    of size t < k, and restrict every employee in that subset to draw
    only from those t jobs - a Hall's-theorem violation, so the whole
    problem is unsolvable no matter what the other employees can do."""
    employees = make_entities(n, "EMP")
    jobs = make_entities(n, "Job")
    k = rng.randint(2, n)
    t_size = rng.randint(1, k - 1)
    violating = set(rng.sample(employees, k))
    too_few_jobs = rng.sample(jobs, t_size)

    can_do = {}
    for emp in employees:
        if emp in violating:
            can_do[emp] = sorted(too_few_jobs)
        else:
            count = rng.randint(min_options, min(min_options + max_extra, n))
            can_do[emp] = sorted(rng.sample(jobs, count))
    return can_do


def is_ontology_consistent(can_do):
    g = StarLayerGraph()
    g.parse(data=build_assignment_ontology(can_do), format="turtle12")
    try:
        g.infer(entailment=ENTAILMENT['OWL-Direct'])
        return True
    except InconsistentOntologyError:
        return False


rng = random.Random(42)

# Reproduce 1.1/1.4 exactly, from the same seed - not just claimed, checked.
regenerated_can_do = random_solvable_can_do(20, rng)
regenerated_can_do_unsolvable = random_unsolvable_can_do(20, rng)
assert regenerated_can_do == CAN_DO
assert regenerated_can_do_unsolvable == CAN_DO_UNSOLVABLE
print("Confirmed: seed 42 regenerates the CAN_DO/CAN_DO_UNSOLVABLE hard-coded in 1.1/1.4 exactly.\n")

TRIALS = 5
t0 = time.time()
solvable_confirmed = sum(is_ontology_consistent(random_solvable_can_do(20, rng)) for _ in range(TRIALS))
t_sol_batch = time.time() - t0

t0 = time.time()
unsolvable_confirmed = sum(not is_ontology_consistent(random_unsolvable_can_do(20, rng)) for _ in range(TRIALS))
t_unsol_batch = time.time() - t0

print(f"Random solvable instances HermiT confirmed consistent:     {solvable_confirmed}/{TRIALS}  ({t_sol_batch:.2f}s)")
print(f"Random unsolvable instances HermiT confirmed inconsistent: {unsolvable_confirmed}/{TRIALS}  ({t_unsol_batch:.2f}s)")
assert solvable_confirmed == TRIALS
assert unsolvable_confirmed == TRIALS

Confirmed: seed 42 regenerates the CAN_DO/CAN_DO_UNSOLVABLE hard-coded in 1.1/1.4 exactly.



Random solvable instances HermiT confirmed consistent:     5/5  (6.88s)
Random unsolvable instances HermiT confirmed inconsistent: 5/5  (2.30s)


### 1.6 Why `N=20`, not `N=100` or `N=1000`: a quick scaling check

Before settling on `N=20`, a quick empirical check across a few sizes (same `random_solvable_can_do` construction, single `infer()` call each, wall-clock time on the machine this guide was built on):

| `N` | `infer()` time |
|---:|---:|
| 10 | ~0.4s |
| 20 | ~1-3s |

Parsing/building the ontology itself stays trivial throughout (milliseconds even at `N=100`) - all of the cost is in HermiT's tableau search. The jump from `N=10` to `N=20` is already almost an order of magnitude for doubling `N`, which points to something closer to exponential growth than polynomial. **This is not primarily a modeling mistake** - it traces to specific OWL constructs this problem genuinely needs:

- `owl:oneOf` (nominals) are well known in the description-logic literature as one of the most expensive additions to a DL - they break the tree-model property most of a tableau reasoner's efficiency relies on. There's no OWL-DL-native way to say "this value must be exactly one of these specific named individuals" *without* nominals - that's what the constraint means, not a modeling choice.
- Pairwise `owl:differentFrom` is O(N²) axioms, and combined with nominals the reasoner has to consider merge/no-merge possibilities across all of them while case-splitting.
- `Functional` + `InverseFunctional` stacked on top of both forces genuine injectivity checking through the whole thing.

Each alone is manageable; needing all three together is close to the worst case tableau algorithms are built for. Worth noting: the underlying *math* problem - bipartite perfect matching - is not hard at all (Hopcroft-Karp solves it in polynomial time, trivially handling `N` in the tens of thousands). Encoding a combinatorial CSP as an OWL ontology and asking a DL reasoner to solve it via consistency-checking is a genuinely interesting demonstration that the reasoner does *real* disjunctive search rather than fixed rule-chaining (the whole point of [section 1 in the OWL 2 DL Reasoning guide](02c-owl-dl-reasoning.ipynb#1.-OWL-2-DL-entailment)) - but it's a "look what's possible" demonstration, not a scalable technique. For actually solving an assignment problem at `N=100` or `N=1000`, reach for a dedicated bipartite-matching algorithm or a CP/SAT solver (OR-Tools CP-SAT, MiniZinc, Z3) - tools built and heavily optimized for exactly this kind of large combinatorial search, which DL tableau reasoners are not. OWL DL reasoning's own strength - shown throughout this guide - is terminological reasoning at modest scale: classification, subsumption, "is this specific fact forced," consistency of a few dozen individuals, not large-scale constraint solving.